In [ ]:
# In[1]:

import pandas as pd
import pytz

# load metric.csv (reusing variable names if rerun in the same kernel)
df = pd.read_csv('metric.csv')

# ensure numeric value and integer timestamp
df['timestamp'] = pd.to_numeric(df['timestamp'], errors='coerce').astype('Int64')
df['value'] = pd.to_numeric(df['value'], errors='coerce')

# convert timestamp to timezone-aware datetime in UTC then to Asia/Shanghai (UTC+8)
df['dt'] = pd.to_datetime(df['timestamp'].astype('int'), unit='s', utc=True).dt.tz_convert(pytz.timezone('Asia/Shanghai'))

# groupby cmdb_id and kpi_name and compute global stats (over the entire series)
grouped = df.groupby(['cmdb_id', 'kpi_name']).agg(
    count=('value', 'count'),
    min_timestamp=('dt', 'min'),
    max_timestamp=('dt', 'max'),
    mean=('value', 'mean'),
    std=('value', 'std'),
    p50=('value', lambda x: x.quantile(0.5)),
    p95=('value', lambda x: x.quantile(0.95)),
    p99=('value', lambda x: x.quantile(0.99))
).reset_index()

# format datetimes to compact strings with timezone offset and round numeric summaries
grouped['min_timestamp'] = grouped['min_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S%z')
grouped['max_timestamp'] = grouped['max_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S%z')
for col in ['mean', 'std', 'p50', 'p95', 'p99']:
    grouped[col] = grouped[col].round(6)

# sort by count descending and show top 20 groups as compact table with requested columns
summary = grouped.sort_values('count', ascending=False)[['cmdb_id', 'kpi_name', 'count', 'min_timestamp', 'max_timestamp', 'mean', 'std', 'p50', 'p95', 'p99']].head(20)

summary

```
Out[1]:
```
```python
# Summarize the previously computed 'summary' DataFrame into a concise plain-English string.
# Reuse 'summary' from the kernel state and produce a compact human-readable conclusion.

rows = summary.head(6)
lines = []
lines.append("Global summary (grouped by cmdb_id and kpi_name) for metric.csv:")
lines.append(f"- Top groups shown: {summary.shape[0]} (displaying up to 20).")
# Determine common time range among displayed groups (they appear identical in this run)
min_ts = summary['min_timestamp'].min()
max_ts = summary['max_timestamp'].max()
lines.append(f"- Time range across these groups: {min_ts} to {max_ts} (Asia/Shanghai, UTC+8).")
lines.append("- All displayed top groups have count = 31 samples.")
lines.append("- Example groups with key statistics (mean, std, p50, p95, p99):")

for r in rows.itertuples(index=False):
    cmdb, kpi, cnt, min_t, max_t, mean, std, p50, p95, p99 = r
    lines.append(f"  • {cmdb} | {kpi} | count={int(cnt)} | mean={mean} | std={std} | p50={p50} | p95={p95} | p99={p99}")

final_summary = "\n".join(lines)
final_summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  count             min_timestamp             max_timestamp          mean           std           p50           p95           p99
6         IG01           JVM-Threads_7778_JVM_ThreadCount_Threads     31  2021-03-23 16:30:00+0800  2021-03-23 17:00:00+0800  7.103226e+01  2.198240e+00  7.100000e+01  7.400000e+01  7.540000e+01
1115  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...     31  2021-03-23 16:30:00+0800  2021-03-23 17:00:00+0800  1.390000e+02  0.000000e+00  1.390000e+02  1.390000e+02  1.390000e+02
1116  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...     31  2021-03-23 16:30:00+0800  2021-03-23 17:00:00+0800  2.304870e+05  0.000000e+00  2.304870e+05  2.304870e+05  2.304870e+05
1117  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_Processin...     31  2021-03-23 16:30:00+0800  2021-03-23 17:00:00+0800  9.675353e+08  1.560511e+06  9.674164e+08  9.699066e+08  9.701466e+08
1118  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...     31  2021-03-23 16:30:00+0800  2021-03-23 17:00:00+0800  1.858847e+06  3.964689e+03  1.858799e+06  1.864753e+06  1.865289e+06
1119  Tomcat04    Tomcat-Sessions_7441--UOCP_SESSIONActiveCounter     31  2021-03-23 16:30:00+0800  2021-03-23 17:00:00+0800  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
1109  Tomcat04                 OSLinux-OSLinux_ZABBIX_Host_Uptime     31  2021-03-23 16:30:00+0800  2021-03-23 17:00:00+0800  7.687976e+06  5.477201e+02  7.687975e+06  7.688789e+06  7.688862e+06
1098  Tomcat04      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies     31  2021-03-23 16:30:00+0800  2021-03-23 17:00:00+0800  1.000000e+00  0.000000e+00  1.000000e+00  1.000000e+00  1.000000e+00
1101  Tomcat04  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...     31  2021-03-23 16:30:00+0800  2021-03-23 17:00:00+0800  3.451610e-01  5.058800e-02  3.000000e-01  4.000000e-01  4.000000e-01
1102  Tomcat04         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount     31  2021-03-23 16:30:00+0800  2021-03-23 17:00:00+0800  3.000000e+00  0.000000e+00  3.000000e+00  3.000000e+00  3.000000e+00
3         IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed     31  2021-03-23 16:30:00+0800  2021-03-23 17:00:00+0800  5.865392e+07  0.000000e+00  5.865392e+07  5.865392e+07  5.865392e+07
56        MG01                        OSLinux-CPU_CPU_CPUidleutil     31  2021-03-23 16:30:00+0800  2021-03-23 17:00:00+0800  7.177356e+01  8.411880e-01  7.188030e+01  7.238230e+01  7.249880e+01
238       MG02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount     31  2021-03-23 16:30:00+0800  2021-03-23 17:00:00+0800  3.000000e+00  0.000000e+00  3.000000e+00  3.000000e+00  3.000000e+00
244       MG02                 OSLinux-OSLinux_ZABBIX_Host_Uptime     31  2021-03-23 16:30:00+0800  2021-03-23 17:00:00+0800  7.688196e+06  5.482189e+02  7.688195e+06  7.689010e+06  7.689083e+06
234       MG02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies     31  2021-03-23 16:30:00+0800  2021-03-23 17:00:00+0800  1.000000e+00  0.000000e+00  1.000000e+00  1.000000e+00  1.000000e+00
237       MG02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...     31  2021-03-23 16:30:00+0800  2021-03-23 17:00:00+0800  3.000000e-01  0.000000e+00  3.000000e-01  3.000000e-01  3.000000e-01
874   Tomcat02  Tomcat-Requests_7441-"http-nio-8003"_Processin...     31  2021-03-23 16:30:00+0800  2021-03-23 17:00:00+0800  1.709423e+09  1.602503e+06  1.709400e+09  1.711828e+09  1.712127e+09
875   Tomcat02  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...     31  2021-03-23 16:30:00+0800  2021-03-23 17:00:00+0800  2.799830e+06  3.986580e+03  2.799788e+06  2.805718e+06  2.806285e+06
876   Tomcat02    Tomcat-Sessions_7441--UOCP_SESSIONActiveCounter     31  2021-03-23 16:30:00+0800  2021-03-23 17:00:00+0800  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
877   Tomcat02  Tomcat-Sessions_7441--UOCP_SESSIONKeepaliveCou...     31  2021-03-23 16:30:00+0800  2021-03-23 17:00:00+0800  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz

# Reuse existing dataframes if present; otherwise load metric.csv
try:
    df
except NameError:
    df = pd.read_csv('metric.csv')
    df['timestamp'] = pd.to_numeric(df['timestamp'], errors='coerce').astype('Int64')
    df['value'] = pd.to_numeric(df['value'], errors='coerce')
    df['dt'] = pd.to_datetime(df['timestamp'].astype('int'), unit='s', utc=True).dt.tz_convert(pytz.timezone('Asia/Shanghai'))

# Ensure 'summary' exists (computed previously)
try:
    summary
except NameError:
    # load/compute summary as in prior step if missing
    df['timestamp'] = pd.to_numeric(df['timestamp'], errors='coerce').astype('Int64')
    df['value'] = pd.to_numeric(df['value'], errors='coerce')
    df['dt'] = pd.to_datetime(df['timestamp'].astype('int'), unit='s', utc=True).dt.tz_convert(pytz.timezone('Asia/Shanghai'))
    summary = df.groupby(['cmdb_id', 'kpi_name']).agg(
        count=('value', 'count'),
        min_timestamp=('dt', 'min'),
        max_timestamp=('dt', 'max'),
        mean=('value', 'mean'),
        std=('value', 'std'),
        p50=('value', lambda x: x.quantile(0.5)),
        p95=('value', lambda x: x.quantile(0.95)),
        p99=('value', lambda x: x.quantile(0.99))
    ).reset_index()
    # round numeric thresholds for stability
    for col in ['mean', 'std', 'p50', 'p95', 'p99']:
        if col in summary.columns:
            summary[col] = summary[col].round(6)

# Prepare thresholds: take p50/p95/p99 from summary
thresholds = summary[['cmdb_id', 'kpi_name', 'p50', 'p95', 'p99']].copy()

# Merge thresholds into metric rows (global thresholds applied before any filtering)
merged = pd.merge(df, thresholds, on=['cmdb_id', 'kpi_name'], how='left', validate='m:1')

# Detect anomalies: value >= group's p95 (ensure numeric comparison; ignore rows missing p95)
metric_anoms = merged[ merged['p95'].notna() & (merged['value'] >= merged['p95']) ].copy()

# Ensure dt exists and is timezone-aware Asia/Shanghai
if 'dt' not in metric_anoms.columns:
    metric_anoms['dt'] = pd.to_datetime(metric_anoms['timestamp'].astype('int'), unit='s', utc=True).dt.tz_convert(pytz.timezone('Asia/Shanghai'))
else:
    metric_anoms['dt'] = pd.to_datetime(metric_anoms['dt']).dt.tz_convert(pytz.timezone('Asia/Shanghai'))

# Aggregate anomalies per group
agg = metric_anoms.groupby(['cmdb_id', 'kpi_name']).agg(
    anomaly_count=('value', 'count'),
    earliest_anomaly_timestamp=('dt', 'min'),
    max_value_in_window=('value', 'max'),
    p50=('p50', 'first'),
    p95=('p95', 'first'),
    p99=('p99', 'first')
).reset_index()

# Format earliest timestamp compactly
agg['earliest_anomaly_timestamp'] = agg['earliest_anomaly_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S%z')

# Sort by anomaly_count descending and limit to top 20
agg = agg.sort_values('anomaly_count', ascending=False)[['cmdb_id', 'kpi_name', 'anomaly_count', 'earliest_anomaly_timestamp', 'max_value_in_window', 'p50', 'p95', 'p99']].head(20)

# Final displayed result (compact)
agg

```
Out[2]:
```
```python
# Summarize anomaly scan results into a concise plain-English string.
# Reuse 'agg' (the aggregated anomalies table) from the kernel state.

try:
    agg
except NameError:
    raise RuntimeError("Aggregated anomalies table 'agg' not found in kernel state.")

total_groups = int(agg.shape[0])
overall_earliest = agg['earliest_anomaly_timestamp'].min() if total_groups > 0 else "N/A"
top_rows = agg.head(5)

lines = []
lines.append(f"Anomaly scan summary (value >= group's p95) — {total_groups} component-KPI groups had ≥1 anomaly.")
lines.append(f"Overall earliest anomaly: {overall_earliest} (Asia/Shanghai, UTC+8).")
lines.append("Top groups by anomaly count (up to 5):")

for r in top_rows.itertuples(index=False):
    cmdb, kpi, cnt, earliest, maxv, p50, p95, p99 = r
    lines.append(f"• {cmdb} | {kpi} | anomalies={int(cnt)} | earliest={earliest} | max_value={maxv}")

lines.append("Note: Full anomaly rows are saved in kernel variable 'metric_anoms' for inspection.")

final_summary = "\n".join(lines)
final_summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  anomaly_count earliest_anomaly_timestamp  max_value_in_window           p50           p95           p99
0       IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed             31   2021-03-23 16:30:00+0800         5.865392e+07  5.865392e+07  5.865392e+07  5.865392e+07
3       MG02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies             31   2021-03-23 16:30:00+0800         1.000000e+00  1.000000e+00  1.000000e+00  1.000000e+00
5       MG02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount             31   2021-03-23 16:30:00+0800         3.000000e+00  3.000000e+00  3.000000e+00  3.000000e+00
4       MG02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...             31   2021-03-23 16:30:00+0800         3.000000e-01  3.000000e-01  3.000000e-01  3.000000e-01
16  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...             31   2021-03-23 16:30:00+0800         2.304870e+05  2.304870e+05  2.304870e+05  2.304870e+05
19  Tomcat04    Tomcat-Sessions_7441--UOCP_SESSIONActiveCounter             31   2021-03-23 16:30:00+0800         0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
10  Tomcat02  Tomcat-Sessions_7441--UOCP_SESSIONKeepaliveCou...             31   2021-03-23 16:30:00+0800         0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
9   Tomcat02    Tomcat-Sessions_7441--UOCP_SESSIONActiveCounter             31   2021-03-23 16:30:00+0800         0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
11  Tomcat04      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies             31   2021-03-23 16:30:00+0800         1.000000e+00  1.000000e+00  1.000000e+00  1.000000e+00
13  Tomcat04         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount             31   2021-03-23 16:30:00+0800         3.000000e+00  3.000000e+00  3.000000e+00  3.000000e+00
15  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...             31   2021-03-23 16:30:00+0800         1.390000e+02  1.390000e+02  1.390000e+02  1.390000e+02
12  Tomcat04  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...             14   2021-03-23 16:31:00+0800         4.000000e-01  3.000000e-01  4.000000e-01  4.000000e-01
1       IG01           JVM-Threads_7778_JVM_ThreadCount_Threads              6   2021-03-23 16:30:00+0800         7.600000e+01  7.100000e+01  7.400000e+01  7.540000e+01
2       MG01                        OSLinux-CPU_CPU_CPUidleutil              2   2021-03-23 16:46:00+0800         7.251581e+01  7.188030e+01  7.238230e+01  7.249880e+01
6       MG02                 OSLinux-OSLinux_ZABBIX_Host_Uptime              2   2021-03-23 16:59:00+0800         7.689101e+06  7.688195e+06  7.689010e+06  7.689083e+06
7   Tomcat02  Tomcat-Requests_7441-"http-nio-8003"_Processin...              2   2021-03-23 16:59:00+0800         1.712188e+09  1.709400e+09  1.711828e+09  1.712127e+09
8   Tomcat02  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...              2   2021-03-23 16:59:00+0800         2.806424e+06  2.799788e+06  2.805718e+06  2.806285e+06
14  Tomcat04                 OSLinux-OSLinux_ZABBIX_Host_Uptime              2   2021-03-23 16:59:00+0800         7.688880e+06  7.687975e+06  7.688789e+06  7.688862e+06
17  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_Processin...              2   2021-03-23 16:59:00+0800         9.702064e+08  9.674164e+08  9.699066e+08  9.701466e+08
18  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...              2   2021-03-23 16:59:00+0800         1.865419e+06  1.858799e+06  1.864753e+06  1.865289e+06```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz

# Reuse metric_anoms and summary from kernel state
try:
    metric_anoms
except NameError:
    raise RuntimeError("Kernel variable 'metric_anoms' not found. Run anomaly detection step first.")

try:
    summary
except NameError:
    raise RuntimeError("Kernel variable 'summary' not found. Global thresholds required.")

# Ensure timestamp and dt columns are present and correctly typed
metric_anoms['timestamp'] = pd.to_numeric(metric_anoms['timestamp'], errors='coerce').astype('Int64')
metric_anoms['value'] = pd.to_numeric(metric_anoms['value'], errors='coerce')
# Ensure dt is timezone-aware Asia/Shanghai
metric_anoms['dt'] = pd.to_datetime(metric_anoms['timestamp'].astype('int'), unit='s', utc=True).dt.tz_convert(pytz.timezone('Asia/Shanghai'))

# Sort for sequence detection
metric_anoms = metric_anoms.sort_values(['cmdb_id', 'kpi_name', 'timestamp']).reset_index(drop=True)

# Compute gap in seconds to previous row within each group
metric_anoms['prev_ts'] = metric_anoms.groupby(['cmdb_id', 'kpi_name'])['timestamp'].shift(1)
metric_anoms['gap'] = metric_anoms['timestamp'] - metric_anoms['prev_ts']

# Mark start of a new sequence when gap is not exactly 60 seconds or previous is NaN
metric_anoms['new_seq'] = (metric_anoms['gap'].isna()) | (metric_anoms['gap'] != 60)

# Sequence number within each group
metric_anoms['seq_num'] = metric_anoms.groupby(['cmdb_id', 'kpi_name'])['new_seq'].cumsum().astype(int)

# Create a compact sequence_id (string) unique per group-seq_num
metric_anoms['sequence_id'] = metric_anoms['cmdb_id'].astype(str) + '||' + metric_anoms['kpi_name'].astype(str) + '||' + metric_anoms['seq_num'].astype(str)

# Save annotated anomaly rows for later inspection as required
metric_fault_rows = metric_anoms.copy()

# Aggregate consecutive sequences
seq_agg = metric_fault_rows.groupby(['cmdb_id', 'kpi_name', 'sequence_id', 'seq_num']).agg(
    seq_start_timestamp=('dt', 'min'),
    seq_end_timestamp=('dt', 'max'),
    seq_length=('value', 'count'),
    max_value_in_sequence=('value', 'max'),
    p50=('p50', 'first'),
    p95=('p95', 'first'),
    p99=('p99', 'first')
).reset_index()

# Format timestamps as compact strings in Asia/Shanghai
seq_agg['seq_start_timestamp'] = seq_agg['seq_start_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S%z')
seq_agg['seq_end_timestamp'] = seq_agg['seq_end_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S%z')

# Select and sort top sequences by seq_length descending, limit to top 20
sequences = seq_agg.sort_values('seq_length', ascending=False)[
    ['cmdb_id', 'kpi_name', 'seq_start_timestamp', 'seq_end_timestamp', 'seq_length', 'max_value_in_sequence', 'p50', 'p95', 'p99']
].head(20)

# Display the compact result (per rules, use variable name to display)
sequences, metric_fault_rows.shape

```
Out[3]:
```
```python
# Summarize consecutive anomaly sequences detected (use existing kernel variables).
try:
    metric_fault_rows
    sequences
except NameError as e:
    raise RuntimeError("Required kernel variables 'metric_fault_rows' or 'sequences' not found.") from e

total_sequences = int(metric_fault_rows['sequence_id'].nunique())
total_anom_rows = int(metric_fault_rows.shape[0])
unique_groups = int(metric_fault_rows[['cmdb_id', 'kpi_name']].drop_duplicates().shape[0])

overall_start = sequences['seq_start_timestamp'].min() if not sequences.empty else "N/A"
overall_end = sequences['seq_end_timestamp'].max() if not sequences.empty else "N/A"

lines = []
lines.append(f"Consecutive anomaly sequences detected: {total_sequences} sequences across {unique_groups} component-KPI groups (total anomaly rows: {total_anom_rows}).")
lines.append(f"Overall sequence time range (from reported sequences): {overall_start} to {overall_end} (Asia/Shanghai, UTC+8).")
lines.append("Top sequences by length (up to 5):")

for r in sequences.head(5).itertuples(index=False):
    cmdb, kpi, start_ts, end_ts, length, maxv, p50, p95, p99 = r
    lines.append(
        f"• {cmdb} | {kpi} | length={int(length)} | {start_ts} → {end_ts} | max={maxv:.6g} | p95={p95:.6g}"
    )

lines.append("Note: full anomaly rows annotated with sequence_id are saved in kernel variable 'metric_fault_rows' for inspection.")

final_summary = "\n".join(lines)
final_summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                                           kpi_name       seq_start_timestamp         seq_end_timestamp  seq_length  max_value_in_sequence           p50           p95           p99
0       IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed  2021-03-23 16:30:00+0800  2021-03-23 17:00:00+0800          31           5.865392e+07  5.865392e+07  5.865392e+07  5.865392e+07
7       MG02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies  2021-03-23 16:30:00+0800  2021-03-23 17:00:00+0800          31           1.000000e+00  1.000000e+00  1.000000e+00  1.000000e+00
8       MG02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...  2021-03-23 16:30:00+0800  2021-03-23 17:00:00+0800          31           3.000000e-01  3.000000e-01  3.000000e-01  3.000000e-01
15  Tomcat04      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies  2021-03-23 16:30:00+0800  2021-03-23 17:00:00+0800          31           1.000000e+00  1.000000e+00  1.000000e+00  1.000000e+00
9       MG02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount  2021-03-23 16:30:00+0800  2021-03-23 17:00:00+0800          31           3.000000e+00  3.000000e+00  3.000000e+00  3.000000e+00
13  Tomcat02    Tomcat-Sessions_7441--UOCP_SESSIONActiveCounter  2021-03-23 16:30:00+0800  2021-03-23 17:00:00+0800          31           0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
14  Tomcat02  Tomcat-Sessions_7441--UOCP_SESSIONKeepaliveCou...  2021-03-23 16:30:00+0800  2021-03-23 17:00:00+0800          31           0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
32  Tomcat04    Tomcat-Sessions_7441--UOCP_SESSIONActiveCounter  2021-03-23 16:30:00+0800  2021-03-23 17:00:00+0800          31           0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
29  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...  2021-03-23 16:30:00+0800  2021-03-23 17:00:00+0800          31           2.304870e+05  2.304870e+05  2.304870e+05  2.304870e+05
26  Tomcat04         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount  2021-03-23 16:30:00+0800  2021-03-23 17:00:00+0800          31           3.000000e+00  3.000000e+00  3.000000e+00  3.000000e+00
28  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...  2021-03-23 16:30:00+0800  2021-03-23 17:00:00+0800          31           1.390000e+02  1.390000e+02  1.390000e+02  1.390000e+02
22  Tomcat04  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...  2021-03-23 16:45:00+0800  2021-03-23 16:47:00+0800           3           4.000000e-01  3.000000e-01  4.000000e-01  4.000000e-01
17  Tomcat04  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...  2021-03-23 16:57:00+0800  2021-03-23 16:59:00+0800           3           4.000000e-01  3.000000e-01  4.000000e-01  4.000000e-01
4       IG01           JVM-Threads_7778_JVM_ThreadCount_Threads  2021-03-23 16:58:00+0800  2021-03-23 17:00:00+0800           3           7.400000e+01  7.100000e+01  7.400000e+01  7.540000e+01
11  Tomcat02  Tomcat-Requests_7441-"http-nio-8003"_Processin...  2021-03-23 16:59:00+0800  2021-03-23 17:00:00+0800           2           1.712188e+09  1.709400e+09  1.711828e+09  1.712127e+09
10      MG02                 OSLinux-OSLinux_ZABBIX_Host_Uptime  2021-03-23 16:59:00+0800  2021-03-23 17:00:00+0800           2           7.689101e+06  7.688195e+06  7.689010e+06  7.689083e+06
12  Tomcat02  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...  2021-03-23 16:59:00+0800  2021-03-23 17:00:00+0800           2           2.806424e+06  2.799788e+06  2.805718e+06  2.806285e+06
31  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...  2021-03-23 16:59:00+0800  2021-03-23 17:00:00+0800           2           1.865419e+06  1.858799e+06  1.864753e+06  1.865289e+06
30  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_Processin...  2021-03-23 16:59:00+0800  2021-03-23 17:00:00+0800           2           9.702064e+08  9.674164e+08  9.699066e+08  9.701466e+08
27  Tomcat04                 OSLinux-OSLinux_ZABBIX_Host_Uptime  2021-03-23 16:59:00+0800  2021-03-23 17:00:00+0800           2           7.688880e+06  7.687975e+06  7.688789e+06  7.688862e+06, (375, 13))```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz

# Load trace.csv (reuse kernel where appropriate)
trace_df = pd.read_csv('trace.csv')

# Ensure types
trace_df['timestamp'] = pd.to_numeric(trace_df['timestamp'], errors='coerce').astype('Int64')
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

# Create timezone-aware datetime in Asia/Shanghai (UTC+8)
tz = pytz.timezone('Asia/Shanghai')
trace_df['dt'] = pd.to_datetime(trace_df['timestamp'].astype('int'), unit='s', utc=True).dt.tz_convert(tz)

# Group and compute global statistics per cmdb_id and trace_name
trace_summary = trace_df.groupby(['cmdb_id', 'trace_name']).agg(
    count=('value', 'count'),
    min_timestamp=('dt', 'min'),
    max_timestamp=('dt', 'max'),
    mean=('value', 'mean'),
    std=('value', 'std'),
    p50=('value', lambda x: x.quantile(0.5)),
    p95=('value', lambda x: x.quantile(0.95)),
    p99=('value', lambda x: x.quantile(0.99))
).reset_index()

# Format timestamps and numeric columns compactly
trace_summary['min_timestamp'] = trace_summary['min_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S%z')
trace_summary['max_timestamp'] = trace_summary['max_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S%z')
for col in ['mean', 'std', 'p50', 'p95', 'p99']:
    trace_summary[col] = trace_summary[col].round(6)

# Sort by count descending and keep top 20 groups
trace_summary = trace_summary.sort_values('count', ascending=False)[
    ['cmdb_id', 'trace_name', 'count', 'min_timestamp', 'max_timestamp', 'mean', 'std', 'p50', 'p95', 'p99']
].head(20)

# Display the compact summary and keep it in kernel variable 'trace_summary'
trace_summary

```
Out[4]:
```
```python
# Summarize the previously computed 'trace_summary' into a concise plain-English string.
# This is a summarization task: produce a compact human-readable conclusion as a string.

try:
    trace_summary
except NameError:
    raise RuntimeError("Kernel variable 'trace_summary' not found. Run the trace summary step first.")

n_groups = int(trace_summary.shape[0])
if n_groups == 0:
    final_summary = "No trace groups found in 'trace_summary'."
else:
    # Determine overall time range across listed groups
    # parse timestamp strings to datetimes to be safe
    min_times = pd.to_datetime(trace_summary['min_timestamp'])
    max_times = pd.to_datetime(trace_summary['max_timestamp'])
    overall_start = min_times.min().strftime('%Y-%m-%d %H:%M:%S%z')
    overall_end = max_times.max().strftime('%Y-%m-%d %H:%M:%S%z')

    lines = []
    lines.append(f"Trace global summary (using full series): {n_groups} cmdb_id/trace_name groups (top 20 shown).")
    lines.append(f"Time range across these groups: {overall_start} to {overall_end} (Asia/Shanghai, UTC+8).")
    lines.append("Most groups have count = 1 in this dataset (per the displayed top groups).")
    lines.append("Examples (up to 5) with key stats (count, mean, p95):")

    for r in trace_summary.head(5).itertuples(index=False):
        cmdb, tname, cnt, min_ts, max_ts, mean, std, p50, p95, p99 = r
        lines.append(f"• {cmdb} | {tname} | count={int(cnt)} | mean={mean} | p95={p95}")

    lines.append("Note: full per-group stats are stored in kernel variable 'trace_summary'.")
    final_summary = "\n".join(lines)

final_summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                       trace_name  count             min_timestamp             max_timestamp          mean  std           p50           p95           p99
263  dockerB2          trace.to_MG02.row_count      1  1970-01-20 01:01:00+0800  1970-01-20 01:01:00+0800   2432.000000  NaN   2432.000000   2432.000000   2432.000000
0        IG01    trace.from_root.duration_mean      1  1970-01-20 01:01:00+0800  1970-01-20 01:01:00+0800      0.646192  NaN      0.646192      0.646192      0.646192
1        IG01     trace.from_root.duration_p95      1  1970-01-20 01:01:00+0800  1970-01-20 01:01:00+0800      2.239000  NaN      2.239000      2.239000      2.239000
2        IG01        trace.from_root.row_count      1  1970-01-20 01:01:00+0800  1970-01-20 01:01:00+0800     73.000000  NaN     73.000000     73.000000     73.000000
3        IG01         trace.self.duration_mean      1  1970-01-20 01:01:00+0800  1970-01-20 01:01:00+0800      0.427832  NaN      0.427832      0.427832      0.427832
4        IG01          trace.self.duration_p95      1  1970-01-20 01:01:00+0800  1970-01-20 01:01:00+0800      2.053000  NaN      2.053000      2.053000      2.053000
5        IG01             trace.self.row_count      1  1970-01-20 01:01:00+0800  1970-01-20 01:01:00+0800  52415.000000  NaN  52415.000000  52415.000000  52415.000000
6        IG01  trace.to_Tomcat01.duration_mean      1  1970-01-20 01:01:00+0800  1970-01-20 01:01:00+0800      0.397128  NaN      0.397128      0.397128      0.397128
224  dockerA2          trace.to_MG01.row_count      1  1970-01-20 01:01:00+0800  1970-01-20 01:01:00+0800   1784.000000  NaN   1784.000000   1784.000000   1784.000000
225  dockerA2      trace.to_MG02.duration_mean      1  1970-01-20 01:01:00+0800  1970-01-20 01:01:00+0800      0.067921  NaN      0.067921      0.067921      0.067921
226  dockerA2       trace.to_MG02.duration_p95      1  1970-01-20 01:01:00+0800  1970-01-20 01:01:00+0800      0.797600  NaN      0.797600      0.797600      0.797600
227  dockerA2          trace.to_MG02.row_count      1  1970-01-20 01:01:00+0800  1970-01-20 01:01:00+0800   1765.000000  NaN   1765.000000   1765.000000   1765.000000
228  dockerB1    trace.from_MG01.duration_mean      1  1970-01-20 01:01:00+0800  1970-01-20 01:01:00+0800      0.365816  NaN      0.365816      0.365816      0.365816
229  dockerB1     trace.from_MG01.duration_p95      1  1970-01-20 01:01:00+0800  1970-01-20 01:01:00+0800      2.090000  NaN      2.090000      2.090000      2.090000
230  dockerB1        trace.from_MG01.row_count      1  1970-01-20 01:01:00+0800  1970-01-20 01:01:00+0800   8515.000000  NaN   8515.000000   8515.000000   8515.000000
231  dockerB1    trace.from_MG02.duration_mean      1  1970-01-20 01:01:00+0800  1970-01-20 01:01:00+0800      0.361490  NaN      0.361490      0.361490      0.361490
216  dockerA2    trace.from_root.duration_mean      1  1970-01-20 01:01:00+0800  1970-01-20 01:01:00+0800      0.030393  NaN      0.030393      0.030393      0.030393
217  dockerA2     trace.from_root.duration_p95      1  1970-01-20 01:01:00+0800  1970-01-20 01:01:00+0800      0.033250  NaN      0.033250      0.033250      0.033250
218  dockerA2        trace.from_root.row_count      1  1970-01-20 01:01:00+0800  1970-01-20 01:01:00+0800    392.000000  NaN    392.000000    392.000000    392.000000
219  dockerA2         trace.self.duration_mean      1  1970-01-20 01:01:00+0800  1970-01-20 01:01:00+0800      0.018035  NaN      0.018035      0.018035      0.018035```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz

# Reuse trace_df and trace_summary if present, otherwise load them
try:
    trace_df
except NameError:
    trace_df = pd.read_csv('trace.csv')

# Ensure correct dtypes
trace_df['timestamp'] = pd.to_numeric(trace_df['timestamp'], errors='coerce').astype('Int64')
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

# Ensure dt column timezone-aware Asia/Shanghai (UTC+8)
tz = pytz.timezone('Asia/Shanghai')
trace_df['dt'] = pd.to_datetime(trace_df['timestamp'].astype('int'), unit='s', utc=True).dt.tz_convert(tz)

# Reuse trace_summary from kernel (it contains global p50/p95/p99 for top groups)
try:
    trace_summary
except NameError:
    raise RuntimeError("Kernel variable 'trace_summary' not found. Please compute global trace_summary first.")

# Prepare thresholds (p50/p95/p99) from trace_summary
thresholds = trace_summary[['cmdb_id', 'trace_name', 'p50', 'p95', 'p99']].copy()

# Merge thresholds into full trace series (global thresholds applied before filtering)
merged = pd.merge(trace_df, thresholds, on=['cmdb_id', 'trace_name'], how='left', validate='m:1')

# Define incident time window in Asia/Shanghai (inclusive)
start = pd.Timestamp('2021-03-23 16:30:00').tz_localize(tz)
end = pd.Timestamp('2021-03-23 17:00:00').tz_localize(tz)

# Filter merged rows to the incident window
windowed = merged[(merged['dt'] >= start) & (merged['dt'] <= end)].copy()

# Detect anomalies: value >= group's p95 (only where p95 is available)
trace_anoms = windowed[ windowed['p95'].notna() & (windowed['value'] >= windowed['p95']) ].copy()

# Aggregate anomalies per trace group
agg = trace_anoms.groupby(['cmdb_id', 'trace_name']).agg(
    anomaly_count=('value', 'count'),
    earliest_anomaly_timestamp=('dt', 'min'),
    max_value_in_window=('value', 'max'),
    p50=('p50', 'first'),
    p95=('p95', 'first'),
    p99=('p99', 'first')
).reset_index()

# Format earliest timestamp compactly
if not agg.empty:
    agg['earliest_anomaly_timestamp'] = agg['earliest_anomaly_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S%z')

# Sort by anomaly_count desc and limit to top 20
trace_agg = agg.sort_values('anomaly_count', ascending=False)[
    ['cmdb_id', 'trace_name', 'anomaly_count', 'earliest_anomaly_timestamp', 'max_value_in_window', 'p50', 'p95', 'p99']
].head(20)

# Save full anomaly rows in kernel variable 'trace_anoms' (already assigned) for later inspection
# Display the compact aggregated table as the final output
trace_agg

```
Out[5]:
```
```python
# Summarize the trace anomaly detection result (use existing kernel vars: trace_agg, trace_anoms)
try:
    trace_agg
    trace_anoms
except NameError:
    raise RuntimeError("Required kernel variables 'trace_agg' or 'trace_anoms' not found. Run the detection step first.")

tz_str = "Asia/Shanghai (UTC+8)"
window_start = "2021-03-23 16:30:00"
window_end = "2021-03-23 17:00:00"

num_groups = int(trace_agg.shape[0])
num_rows = int(trace_anoms.shape[0])

if num_groups == 0:
    final_summary = (
        f"No trace-group anomalies were detected within the incident window "
        f"{window_start} to {window_end} {tz_str} using each group's global p95 threshold.\n"
        f"Anomaly groups: {num_groups}. Anomalous rows: {num_rows}.\n"
        "The kernel variable 'trace_anoms' contains the (empty) filtered anomaly rows for inspection."
    )
else:
    top_sample = trace_agg.head(5)
    lines = [
        f"Trace-group anomaly summary for window {window_start} to {window_end} {tz_str}:",
        f"- Number of trace groups with ≥1 anomaly: {num_groups}",
        f"- Total anomalous trace rows: {num_rows}",
        "- Top groups (up to 5) with key fields:"
    ]
    for r in top_sample.itertuples(index=False):
        cmdb, tname, cnt, earliest, maxv, p50, p95, p99 = r
        lines.append(f"  • {cmdb} | {tname} | anomalies={int(cnt)} | earliest={earliest} | max_value={maxv}")
    lines.append("Full anomalous rows are saved in kernel variable 'trace_anoms' for inspection.")
    final_summary = "\n".join(lines)

final_summary
```

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [cmdb_id, trace_name, anomaly_count, earliest_anomaly_timestamp, max_value_in_window, p50, p95, p99]
Index: []```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz

# Load log.csv
log_df = pd.read_csv('log.csv')

# Ensure correct dtypes
log_df['timestamp'] = pd.to_numeric(log_df['timestamp'], errors='coerce').astype('Int64')
log_df['value'] = pd.to_numeric(log_df['value'], errors='coerce')

# Create timezone-aware datetime in Asia/Shanghai (UTC+8)
tz = pytz.timezone('Asia/Shanghai')
log_df['dt'] = pd.to_datetime(log_df['timestamp'].astype('int'), unit='s', utc=True).dt.tz_convert(tz)

# Group and compute global statistics per cmdb_id and log_name
log_summary = log_df.groupby(['cmdb_id', 'log_name']).agg(
    count=('value', 'count'),
    min_timestamp=('dt', 'min'),
    max_timestamp=('dt', 'max'),
    mean=('value', 'mean'),
    std=('value', 'std'),
    p50=('value', lambda x: x.quantile(0.5)),
    p95=('value', lambda x: x.quantile(0.95)),
    p99=('value', lambda x: x.quantile(0.99))
).reset_index()

# Format timestamps and numeric columns compactly
log_summary['min_timestamp'] = log_summary['min_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S%z')
log_summary['max_timestamp'] = log_summary['max_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S%z')
for col in ['mean', 'std', 'p50', 'p95', 'p99']:
    log_summary[col] = log_summary[col].round(6)

# Prepare top-20 compact table sorted by count descending for display
log_summary_top20 = log_summary.sort_values('count', ascending=False)[
    ['cmdb_id', 'log_name', 'count', 'min_timestamp', 'max_timestamp', 'mean', 'std', 'p50', 'p95', 'p99']
].head(20)

# 'log_summary' (full aggregated result) is saved in the kernel as requested.
# Display the compact top-20 summary (per environment rules, show variable)
log_summary_top20

```
Out[6]:
```
```python
# Summarize the previously computed 'log_summary' (and top-20 view) into a concise plain-English string.
try:
    log_summary_top20
except NameError:
    try:
        log_summary_top20 = log_summary.sort_values('count', ascending=False).head(20)
    except NameError:
        raise RuntimeError("Kernel variable 'log_summary' not found. Run the log summary step first.")

n_groups = int(log_summary_top20.shape[0])
overall_start = log_summary_top20['min_timestamp'].min() if n_groups>0 else "N/A"
overall_end = log_summary_top20['max_timestamp'].max() if n_groups>0 else "N/A"

lines = []
lines.append(f"Log global summary (full series aggregated by cmdb_id and log_name). Top {n_groups} groups shown.")
lines.append(f"Time range across these groups: {overall_start} to {overall_end} (Asia/Shanghai, UTC+8).")
lines.append("Most top groups have 31 samples (one per minute over the 31-minute window).")
lines.append("Notable examples from the top groups:")
# pick a few representative rows if present
examples = log_summary_top20.head(6)
for r in examples.itertuples(index=False):
    cmdb, lname, cnt, min_ts, max_ts, mean, std, p50, p95, p99 = r
    lines.append(f"• {cmdb} | {lname} | count={int(cnt)} | mean={mean} | p95={p95}")

lines.append("Highlights:")
lines.append("• Several Tomcat instances (Tomcat01/02/03/04) show log.row_count ~420-430 (p95 ~471–515).")
lines.append("• apache01/apache02 log.row_count are higher (~846–848 mean, p95 ~968).")
lines.append("• Mysql01 has fewer samples (count=28) with low log.row_count mean (~5.29, p95=12).")
lines.append("• GC pause metrics (log.gc_pause_max) are present with counts ~20–23 and p95 in the ~0.08–0.13 range.")
lines.append("The full aggregated per-group results are stored in kernel variable 'log_summary' for inspection.")

final_summary = "\n".join(lines)
final_summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id          log_name  count             min_timestamp             max_timestamp        mean         std      p50       p95       p99
4   Tomcat01     log.row_count     31  2021-03-23 16:30:00+0800  2021-03-23 17:00:00+0800  420.064516   81.576114  434.000  484.5000  485.0000
2   Tomcat01   log.error_count     31  2021-03-23 16:30:00+0800  2021-03-23 17:00:00+0800    0.000000    0.000000    0.000    0.0000    0.0000
16  apache02   log.error_count     31  2021-03-23 16:30:00+0800  2021-03-23 17:00:00+0800    0.000000    0.000000    0.000    0.0000    0.0000
15  apache01     log.row_count     31  2021-03-23 16:30:00+0800  2021-03-23 17:00:00+0800  848.483871  162.897692  870.000  968.0000  977.2000
5   Tomcat02   log.error_count     31  2021-03-23 16:30:00+0800  2021-03-23 17:00:00+0800    0.000000    0.000000    0.000    0.0000    0.0000
7   Tomcat02     log.row_count     31  2021-03-23 16:30:00+0800  2021-03-23 17:00:00+0800  424.290323   85.208057  432.000  503.0000  515.0000
8   Tomcat03   log.error_count     31  2021-03-23 16:30:00+0800  2021-03-23 17:00:00+0800    0.000000    0.000000    0.000    0.0000    0.0000
13  Tomcat04     log.row_count     31  2021-03-23 16:30:00+0800  2021-03-23 17:00:00+0800  422.935484   81.475123  431.000  471.0000  492.3000
11  Tomcat04   log.error_count     31  2021-03-23 16:30:00+0800  2021-03-23 17:00:00+0800    0.000000    0.000000    0.000    0.0000    0.0000
10  Tomcat03     log.row_count     31  2021-03-23 16:30:00+0800  2021-03-23 17:00:00+0800  430.419355   85.960756  439.000  502.5000  507.9000
17  apache02     log.row_count     31  2021-03-23 16:30:00+0800  2021-03-23 17:00:00+0800  846.451613  163.093601  873.000  968.5000  973.1000
14  apache01   log.error_count     31  2021-03-23 16:30:00+0800  2021-03-23 17:00:00+0800    0.000000    0.000000    0.000    0.0000    0.0000
1    Mysql01     log.row_count     28  2021-03-23 16:30:00+0800  2021-03-23 17:00:00+0800    5.285714    5.408205    4.000   12.0000   22.9500
0    Mysql01   log.error_count     28  2021-03-23 16:30:00+0800  2021-03-23 17:00:00+0800    0.000000    0.000000    0.000    0.0000    0.0000
12  Tomcat04  log.gc_pause_max     23  2021-03-23 16:30:00+0800  2021-03-23 16:59:00+0800    0.064348    0.028576    0.070    0.1170    0.1278
9   Tomcat03  log.gc_pause_max     22  2021-03-23 16:30:00+0800  2021-03-23 17:00:00+0800    0.054091    0.025006    0.050    0.0800    0.1195
6   Tomcat02  log.gc_pause_max     21  2021-03-23 16:30:00+0800  2021-03-23 16:59:00+0800    0.057619    0.028619    0.060    0.0900    0.1300
3   Tomcat01  log.gc_pause_max     20  2021-03-23 16:31:00+0800  2021-03-23 16:58:00+0800    0.060000    0.025752    0.065    0.0905    0.0981```
```

In [ ]:
# In[7]:

import pandas as pd
import pytz

# Reuse existing log_df and log_summary from kernel state
try:
    log_df
except NameError:
    log_df = pd.read_csv('log.csv')

try:
    log_summary
except NameError:
    raise RuntimeError("Kernel variable 'log_summary' not found. Compute global log_summary first.")

# Ensure types and timezone-aware datetime (Asia/Shanghai UTC+8)
tz = pytz.timezone('Asia/Shanghai')
log_df['timestamp'] = pd.to_numeric(log_df['timestamp'], errors='coerce').astype('Int64')
log_df['value'] = pd.to_numeric(log_df['value'], errors='coerce')
log_df['dt'] = pd.to_datetime(log_df['timestamp'].astype('int'), unit='s', utc=True).dt.tz_convert(tz)

# Prepare thresholds (global p50/p95/p99) from log_summary
thresholds = log_summary[['cmdb_id', 'log_name', 'p50', 'p95', 'p99']].copy()

# Merge thresholds into the full log series (validate m:1)
merged = pd.merge(log_df, thresholds, on=['cmdb_id', 'log_name'], how='left', validate='m:1')

# Define incident window (inclusive) in Asia/Shanghai
start = pd.Timestamp('2021-03-23 16:30:00').tz_localize(tz)
end = pd.Timestamp('2021-03-23 17:00:00').tz_localize(tz)

# Filter to the incident window
windowed = merged[(merged['dt'] >= start) & (merged['dt'] <= end)].copy()

# Detect anomalies where value >= group's p95 (only where p95 is available)
log_anoms = windowed[ windowed['p95'].notna() & (windowed['value'] >= windowed['p95']) ].copy()

# Aggregate anomalies per group
agg = log_anoms.groupby(['cmdb_id', 'log_name']).agg(
    anomaly_count=('value', 'count'),
    earliest_anomaly_timestamp=('dt', 'min'),
    max_value_in_window=('value', 'max'),
    p50=('p50', 'first'),
    p95=('p95', 'first'),
    p99=('p99', 'first')
).reset_index()

# Format earliest timestamp compactly
if not agg.empty:
    agg['earliest_anomaly_timestamp'] = agg['earliest_anomaly_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S%z')

# Sort by anomaly_count descending and limit to top 20
log_anom_summary = agg.sort_values('anomaly_count', ascending=False)[
    ['cmdb_id', 'log_name', 'anomaly_count', 'earliest_anomaly_timestamp', 'max_value_in_window', 'p50', 'p95', 'p99']
].head(20)

# Save full anomaly rows for later inspection into kernel variable 'log_anoms' (already assigned)
# Display the compact aggregate table
log_anom_summary

```
Out[7]:
```
```python
# Summarize the log anomaly aggregation into a concise plain-English string.
try:
    log_anom_summary
    log_anoms
except NameError:
    raise RuntimeError("Required kernel variables 'log_anom_summary' or 'log_anoms' not found. Run the anomaly detection step first.")

import pandas as pd

num_groups = int(log_anom_summary.shape[0])
num_rows = int(log_anoms.shape[0])
window = "2021-03-23 16:30:00 → 2021-03-23 17:00:00 (Asia/Shanghai)"

# overall earliest anomaly timestamp if available
if num_groups > 0:
    earliest = pd.to_datetime(log_anom_summary['earliest_anomaly_timestamp']).min().strftime('%Y-%m-%d %H:%M:%S%z')
else:
    earliest = "N/A"

lines = []
lines.append(f"Log anomaly detection summary for window {window}:")
lines.append(f"- Log groups with ≥1 anomaly: {num_groups}")
lines.append(f"- Total anomalous log rows: {num_rows}")
lines.append(f"- Earliest anomaly observed: {earliest}")
lines.append("")

# Note about many error_count groups
lines.append("- Observations:")
# Identify top groups to report (up to 6)
top = log_anom_summary.head(6)
for r in top.itertuples(index=False):
    cmdb, lname, cnt, earliest_ts, maxv, p50, p95, p99 = r
    lines.append(f"  • {cmdb} | {lname} | anomalies={int(cnt)} | earliest={earliest_ts} | max_value={maxv} | p95={p95}")

lines.append("")
lines.append("- Notable points:")
lines.append("  • Several 'log.error_count' groups (multiple Tomcat and apache instances, and Mysql01) show anomaly_count = full-sample counts (e.g., 31), but max_value = 0 and p95 = 0.0 — these were flagged only because the group's p95 threshold is 0 (no actual error events).")
lines.append("  • Some 'log.row_count' groups on Tomcat and apache show occasional high counts exceeding their p95 (examples: Tomcat02, Tomcat04, apache01/apache02).")
lines.append("  • GC pause anomalies (log.gc_pause_max) appear for some Tomcat instances with small counts and max values ~0.10–0.14s, above their p95 thresholds.")
lines.append("")
lines.append("Full anomalous log rows (filtered rows that breached p95) are saved in kernel variable 'log_anoms' for detailed inspection.")

final_summary = "\n".join(lines)
final_summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id          log_name  anomaly_count earliest_anomaly_timestamp  max_value_in_window      p50       p95       p99
5   Tomcat02   log.error_count             31   2021-03-23 16:30:00+0800                 0.00    0.000    0.0000    0.0000
2   Tomcat01   log.error_count             31   2021-03-23 16:30:00+0800                 0.00    0.000    0.0000    0.0000
14  apache01   log.error_count             31   2021-03-23 16:30:00+0800                 0.00    0.000    0.0000    0.0000
16  apache02   log.error_count             31   2021-03-23 16:30:00+0800                 0.00    0.000    0.0000    0.0000
8   Tomcat03   log.error_count             31   2021-03-23 16:30:00+0800                 0.00    0.000    0.0000    0.0000
11  Tomcat04   log.error_count             31   2021-03-23 16:30:00+0800                 0.00    0.000    0.0000    0.0000
0    Mysql01   log.error_count             28   2021-03-23 16:30:00+0800                 0.00    0.000    0.0000    0.0000
1    Mysql01     log.row_count              3   2021-03-23 16:30:00+0800                27.00    4.000   12.0000   22.9500
9   Tomcat03  log.gc_pause_max              3   2021-03-23 16:32:00+0800                 0.13    0.050    0.0800    0.1195
7   Tomcat02     log.row_count              2   2021-03-23 16:36:00+0800               518.00  432.000  503.0000  515.0000
13  Tomcat04     log.row_count              2   2021-03-23 16:48:00+0800               501.00  431.000  471.0000  492.3000
6   Tomcat02  log.gc_pause_max              2   2021-03-23 16:37:00+0800                 0.14    0.060    0.0900    0.1300
4   Tomcat01     log.row_count              2   2021-03-23 16:39:00+0800               485.00  434.000  484.5000  485.0000
10  Tomcat03     log.row_count              2   2021-03-23 16:36:00+0800               510.00  439.000  502.5000  507.9000
15  apache01     log.row_count              2   2021-03-23 16:36:00+0800               979.00  870.000  968.0000  977.2000
12  Tomcat04  log.gc_pause_max              2   2021-03-23 16:46:00+0800                 0.13    0.070    0.1170    0.1278
17  apache02     log.row_count              2   2021-03-23 16:36:00+0800               974.00  873.000  968.5000  973.1000
3   Tomcat01  log.gc_pause_max              1   2021-03-23 16:34:00+0800                 0.10    0.065    0.0905    0.0981```
```

In [ ]:
# In[8]:

import pandas as pd
import numpy as np
import pytz

# Reuse 'sequences' from kernel state
try:
    sequences
except NameError:
    raise RuntimeError("Kernel variable 'sequences' not found. Run sequence detection step first.")

# Ensure numeric columns
sequences['seq_length'] = pd.to_numeric(sequences['seq_length'], errors='coerce').astype('Int64')
sequences['max_value_in_sequence'] = pd.to_numeric(sequences['max_value_in_sequence'], errors='coerce')
sequences['p95'] = pd.to_numeric(sequences['p95'], errors='coerce')

# Parse seq_start_timestamp to timezone-aware datetimes in Asia/Shanghai
tz = pytz.timezone('Asia/Shanghai')
# seq_start_timestamp strings include offset; parse then convert to Asia/Shanghai to normalize
sequences['seq_start_dt'] = pd.to_datetime(sequences['seq_start_timestamp']).dt.tz_convert(tz)

# Compute relative severity per sequence
def compute_rel_sev(max_val, p95):
    if pd.isna(p95) or p95 == 0:
        # if p95==0 and max_val>0 -> treat as infinity, else 0
        return np.inf if (not pd.isna(max_val) and max_val > 0) else 0.0
    else:
        return (max_val - p95) / p95

sequences['rel_severity'] = sequences.apply(
    lambda r: compute_rel_sev(r['max_value_in_sequence'], r['p95']), axis=1
)

# Aggregate per component (cmdb_id)
comp_agg = sequences.groupby('cmdb_id').agg(
    number_of_faulty_kpis = ('kpi_name', lambda x: x.nunique()),
    total_fault_points = ('seq_length', 'sum'),
    earliest_fault_start_dt = ('seq_start_dt', 'min'),
    max_relative_severity = ('rel_severity', 'max')
).reset_index()

# Format earliest timestamp compactly (Asia/Shanghai)
comp_agg['earliest_fault_start'] = comp_agg['earliest_fault_start_dt'].dt.strftime('%Y-%m-%d %H:%M:%S%z')

# Replace infinite with a large numeric representation string "inf" for readability, but keep as numeric if desired.
# Here we keep numeric np.inf; sorting will place inf at top.
component_fault_summary = comp_agg[['cmdb_id', 'number_of_faulty_kpis', 'total_fault_points', 'earliest_fault_start', 'max_relative_severity']].sort_values(
    'max_relative_severity', ascending=False
).head(20)

# Display the compact result and keep it in kernel variable 'component_fault_summary'
component_fault_summary

```
Out[8]:
```
```python
# Summarize the per-component fault summary into a concise plain-English string.
try:
    component_fault_summary
except NameError:
    raise RuntimeError("Kernel variable 'component_fault_summary' not found. Run the aggregation step first.")

lines = []
lines.append("Per-component fault summary (top components by relative severity):")
for r in component_fault_summary.itertuples(index=False):
    cmdb, num_kpis, total_pts, earliest, sev = r
    sev_str = ("inf" if (pd.isna(sev) and False) else (f"{sev:.6g}" if (not pd.isna(sev) and sev != float('inf')) else "inf" if sev==float('inf') else "0"))
    lines.append(f"• {cmdb}: faulty_kpis={int(num_kpis)}, total_fault_points={int(total_pts)}, earliest_start={earliest}, max_relative_severity={sev_str}")

final_summary = "\n".join(lines)
final_summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id  number_of_faulty_kpis  total_fault_points      earliest_fault_start  max_relative_severity
3  Tomcat04                      9                 167  2021-03-23 16:30:00+0800               0.000357
2  Tomcat02                      4                  66  2021-03-23 16:30:00+0800               0.000251
1      MG02                      4                  95  2021-03-23 16:30:00+0800               0.000012
0      IG01                      2                  34  2021-03-23 16:30:00+0800               0.000000```
```

In [ ]:
# In[9]:

import pandas as pd
import pytz

# Load error_logs.csv (reuse if already in kernel)
try:
    error_logs_df
except NameError:
    error_logs_df = pd.read_csv('error_logs.csv')

# Ensure types and timezone-aware datetime in Asia/Shanghai (UTC+8)
tz = pytz.timezone('Asia/Shanghai')
error_logs_df['timestamp'] = pd.to_numeric(error_logs_df['timestamp'], errors='coerce').astype('Int64')
error_logs_df['dt'] = pd.to_datetime(error_logs_df['timestamp'].astype('int'), unit='s', utc=True).dt.tz_convert(tz)

# Define incident window (inclusive) in Asia/Shanghai
start = pd.Timestamp('2021-03-23 16:30:00').tz_localize(tz)
end = pd.Timestamp('2021-03-23 17:00:00').tz_localize(tz)

# Filter for Tomcat04 within the window
tomcat04_error_logs = error_logs_df[
    (error_logs_df['cmdb_id'] == 'Tomcat04') &
    (error_logs_df['dt'] >= start) &
    (error_logs_df['dt'] <= end)
].copy()

# Prepare a compact view for display with formatted timestamp string and selected columns
tomcat04_error_logs_view = tomcat04_error_logs.copy()
tomcat04_error_logs_view['timestamp'] = tomcat04_error_logs_view['dt'].dt.strftime('%Y-%m-%d %H:%M:%S%z')
tomcat04_error_logs_view = tomcat04_error_logs_view[['timestamp', 'cmdb_id', 'message']].head(20)

# Save full filtered results in kernel variable 'tomcat04_error_logs' (as required)
# tomcat04_error_logs already set above.

# Display the compact view (per environment rules, use variable name)
tomcat04_error_logs_view, tomcat04_error_logs.shape

```
Out[9]:
```
No error-log entries were found for Tomcat04 in the window 2021-03-23 16:30:00 → 2021-03-23 17:00:00 (Asia/Shanghai).  
The kernel variable 'tomcat04_error_logs' was saved and contains 0 rows (shape: (0, 4)).

The original code execution output of IPython Kernel is also provided below for reference:

(Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [], (0, 4))```
```

In [ ]:
# In[10]:

import pandas as pd
import pytz

# Reuse existing 'log_anoms' from kernel
try:
    log_anoms
except NameError:
    raise RuntimeError("Kernel variable 'log_anoms' not found. Run the log anomaly detection step first.")

# Ensure dt is timezone-aware Asia/Shanghai (UTC+8)
tz = pytz.timezone('Asia/Shanghai')
if 'dt' not in log_anoms.columns:
    log_anoms['dt'] = pd.to_datetime(log_anoms['timestamp'].astype('int'), unit='s', utc=True).dt.tz_convert(tz)
else:
    # parse/convert safely
    log_anoms['dt'] = pd.to_datetime(log_anoms['dt']).dt.tz_convert(tz)

# Define incident window (inclusive) in Asia/Shanghai
start = pd.Timestamp('2021-03-23 16:30:00').tz_localize(tz)
end = pd.Timestamp('2021-03-23 17:00:00').tz_localize(tz)

# Filter anomalies for Tomcat04 within the window
tomcat04_log_anoms = log_anoms[
    (log_anoms['cmdb_id'] == 'Tomcat04') &
    (log_anoms['dt'] >= start) &
    (log_anoms['dt'] <= end)
].copy()

# Prepare compact view (up to 20 rows) with requested columns and formatted timestamp
tomcat04_view = tomcat04_log_anoms.copy()
tomcat04_view['timestamp'] = tomcat04_view['dt'].dt.strftime('%Y-%m-%d %H:%M:%S%z')
tomcat04_view = tomcat04_view[['timestamp', 'cmdb_id', 'log_name', 'value']].head(20)

# Display the compact result and keep full filtered rows in kernel variable 'tomcat04_log_anoms'
tomcat04_view, tomcat04_log_anoms.shape

```
Out[10]:
```
```python
# Summarize Tomcat04 anomalous log rows (use kernel variable 'tomcat04_log_anoms')
try:
    tomcat04_log_anoms
except NameError:
    raise RuntimeError("Kernel variable 'tomcat04_log_anoms' not found. Run the extraction step first.")

# Ensure datetime column
import pandas as pd
tz = 'Asia/Shanghai'
if 'dt' not in tomcat04_log_anoms.columns:
    tomcat04_log_anoms['dt'] = pd.to_datetime(tomcat04_log_anoms['timestamp'].astype(int), unit='s', utc=True).dt.tz_convert(tz)
else:
    tomcat04_log_anoms['dt'] = pd.to_datetime(tomcat04_log_anoms['dt']).dt.tz_convert(tz)

total_rows = int(tomcat04_log_anoms.shape[0])
earliest = tomcat04_log_anoms['dt'].min().strftime('%Y-%m-%d %H:%M:%S%z') if total_rows>0 else "N/A"
latest = tomcat04_log_anoms['dt'].max().strftime('%Y-%m-%d %H:%M:%S%z') if total_rows>0 else "N/A"
by_log = tomcat04_log_anoms.groupby('log_name').agg(
    count=('value','count'),
    max_value=('value','max'),
    max_ts=('dt', 'idxmax')  # index of max for timestamp lookup
).reset_index()

# Find timestamp(s) of max for each log_name (safe lookup)
max_ts_info = {}
for _, row in by_log.iterrows():
    name = row['log_name']
    idx = row['max_ts']
    if pd.notna(idx):
        ts = tomcat04_log_anoms.loc[idx, 'dt'].strftime('%Y-%m-%d %H:%M:%S%z')
    else:
        ts = "N/A"
    max_ts_info[name] = ts

lines = []
lines.append(f"Tomcat04 log anomalies within 2021-03-23 16:30:00 → 2021-03-23 17:00:00 (Asia/Shanghai):")
lines.append(f"- Total anomalous log rows: {total_rows}")
lines.append(f"- Time range (observed): {earliest} → {latest}")
lines.append("- Breakdown by log_name:")

for _, row in by_log.iterrows():
    name = row['log_name']
    cnt = int(row['count'])
    maxv = row['max_value']
    maxts = max_ts_info.get(name, "N/A")
    lines.append(f"  • {name}: {cnt} rows; max_value={maxv} (at {maxts})")

lines.append("")
lines.append("Notes:")
lines.append("- Most flagged rows are 'log.error_count' with value 0. These were marked because the group's global p95 is 0, so value >= p95 held even though no actual errors occurred.")
if 'log.gc_pause_max' in max_ts_info:
    lines.append(f"- One notable real anomaly: 'log.gc_pause_max' reached {by_log.loc[by_log['log_name']=='log.gc_pause_max','max_value'].values[0]:.2f} at {max_ts_info['log.gc_pause_max']}, exceeding its p95.")
lines.append("- The full filtered anomalous rows are saved in kernel variable 'tomcat04_log_anoms' for inspection.")

final_summary = "\n".join(lines)
final_summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                    timestamp   cmdb_id          log_name  value
10   2021-03-23 16:30:00+0800  Tomcat04   log.error_count   0.00
27   2021-03-23 16:31:00+0800  Tomcat04   log.error_count   0.00
45   2021-03-23 16:32:00+0800  Tomcat04   log.error_count   0.00
61   2021-03-23 16:33:00+0800  Tomcat04   log.error_count   0.00
77   2021-03-23 16:34:00+0800  Tomcat04   log.error_count   0.00
95   2021-03-23 16:35:00+0800  Tomcat04   log.error_count   0.00
112  2021-03-23 16:36:00+0800  Tomcat04   log.error_count   0.00
129  2021-03-23 16:37:00+0800  Tomcat04   log.error_count   0.00
145  2021-03-23 16:38:00+0800  Tomcat04   log.error_count   0.00
163  2021-03-23 16:39:00+0800  Tomcat04   log.error_count   0.00
179  2021-03-23 16:40:00+0800  Tomcat04   log.error_count   0.00
193  2021-03-23 16:41:00+0800  Tomcat04   log.error_count   0.00
211  2021-03-23 16:42:00+0800  Tomcat04   log.error_count   0.00
227  2021-03-23 16:43:00+0800  Tomcat04   log.error_count   0.00
244  2021-03-23 16:44:00+0800  Tomcat04   log.error_count   0.00
261  2021-03-23 16:45:00+0800  Tomcat04   log.error_count   0.00
277  2021-03-23 16:46:00+0800  Tomcat04   log.error_count   0.00
278  2021-03-23 16:46:00+0800  Tomcat04  log.gc_pause_max   0.13
294  2021-03-23 16:47:00+0800  Tomcat04   log.error_count   0.00
311  2021-03-23 16:48:00+0800  Tomcat04   log.error_count   0.00, (35, 8))```
```